# Guradian of truth (Baseline)

In [2]:
import time
import torch
import numpy as np
import pandas as pd

from tqdm import tqdm
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from dataclasses import dataclass
from sklearn.metrics import average_precision_score

from transformers import AutoModelForCausalLM, AutoTokenizer

## 1. Загрузка модели

In [2]:
MODEL_DIR = Path.home() / "models" / "GigaChat3-10B-A1.8B-bf16"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    device_map='auto',
    dtype=torch.bfloat16,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
)

device = next(model.parameters()).device
print(f"Model on: {device}")

`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1


Loading weights: 100%|██████████| 363/363 [00:02<00:00, 152.69it/s]
DeepseekV3ForCausalLM LOAD REPORT from: /home/jovyan/models/GigaChat3-10B-A1.8B-bf16
Key                                                 | Status     |  | 
----------------------------------------------------+------------+--+-
model.layers.26.self_attn.kv_b_proj.weight          | UNEXPECTED |  | 
model.layers.26.mlp.gate.weight                     | UNEXPECTED |  | 
model.layers.26.shared_head.norm.weight             | UNEXPECTED |  | 
model.layers.26.enorm.weight                        | UNEXPECTED |  | 
model.layers.26.self_attn.q_proj.weight             | UNEXPECTED |  | 
model.layers.26.shared_head.head.weight             | UNEXPECTED |  | 
model.layers.26.eh_proj.weight                      | UNEXPECTED |  | 
model.layers.26.mlp.shared_experts.gate_proj.weight | UNEXPECTED |  | 
model.layers.26.mlp.experts.gate_up_proj            | UNEXPECTED |  | 
model.layers.26.mlp.gate.e_score_correction_bias    | UNEXPECTED |

Model on: cuda:0


In [50]:
DATA_PATH = '../data/knowledge_bench_public.csv'

df = pd.read_csv(DATA_PATH)
df_train = df.sample(frac=0.7, random_state=42)
df_test = df.drop(df_train.index)

## 2. Извлечение фичей

Для каждого примера выполняется один forward pass с хуками `with accumulator:`

Затем на соснове взятых проб считаются статистики которые играют роль фичей `FeatureAccumulator.compute_features()`.

In [51]:
PROBE_LAYERS = [0, 5, 10, 15, 20, 25]

class FeatureAccumulator:
    """Collects raw tensors from hooks and computes final feature vectors."""

    def __init__(self, model, probe_layers: list[int] = PROBE_LAYERS):
        self.model = model
        self.probe_layers = probe_layers
        self._hooks: list[torch.utils.hooks.RemovableHook] = []
        self._hidden: dict[str, torch.Tensor] = {}

    def attach(self):
        self._hidden.clear()
        for idx in self.probe_layers:
            name = f"layer_{idx}"

            def _make(n):
                def _fn(_, __, out):
                    h = out[0] if isinstance(out, tuple) else out
                    self._hidden[n] = h.detach()
                return _fn

            self._hooks.append(
                self.model.model.layers[idx].register_forward_hook(_make(name))
            )

    def detach(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()

    def __enter__(self):
        self.attach()
        return self

    def __exit__(self, *_):
        self.detach()

    def compute_features(
        self,
        logits: torch.Tensor,
        input_ids: torch.Tensor,
        answer_start: int,
    ) -> dict:
        """
        Args:
            logits:      [1, seq_len, vocab] from model output
            input_ids:   [1, seq_len]
            answer_start: index where answer tokens begin
        Returns:
            dict of numpy arrays ready for classifier
        """
        ####################################
        #       uncertainty features       #
        ####################################
        seq_len = input_ids.shape[1]
        n_answer = seq_len - answer_start

        answer_logits = logits[0, answer_start - 1: seq_len - 1, :].float()
        answer_ids = input_ids[0, answer_start:seq_len]
        log_probs = torch.log_softmax(answer_logits, dim=-1)
        token_lp = log_probs.gather(1, answer_ids.unsqueeze(1)).squeeze(-1)

        probs = torch.softmax(answer_logits, dim=-1)
        entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1)
        top1 = probs.max(dim=-1).values
        top5 = probs.topk(min(5, probs.shape[-1]), dim=-1).values.sum(dim=-1)

        uncertainty_arr = np.array([
            token_lp.mean().item(),
            token_lp.min().item(),
            token_lp.max().item(),
            token_lp.std().item() if n_answer > 1 else 0.0,
            entropy.mean().item(),
            entropy.max().item(),
            entropy.std().item() if n_answer > 1 else 0.0,
            torch.exp(-token_lp.mean()).item(),
            float(n_answer),
            token_lp[0].item(),
            top1.mean().item(),
            top5.mean().item(),
        ], dtype=np.float32)

        ###################################
        #        internal features        #
        ###################################
        last_hs = self._hidden[f"layer_{self.probe_layers[-1]}"][0]
        probe_vector = last_hs[answer_start - 1].cpu().float().numpy()

        int_scalars = []
        for idx in self.probe_layers:
            hs = self._hidden[f"layer_{idx}"][0]
            int_scalars.append(hs[answer_start - 1].norm().item())
            int_scalars.append(hs[answer_start: seq_len].norm(dim=-1).mean().item())

            ans_hs = hs[answer_start - 1: seq_len - 1].unsqueeze(0)
            with torch.no_grad():
                ll = self.model.lm_head(self.model.model.norm(ans_hs)).float()
            ll_p = torch.softmax(ll[0], dim=-1)
            ll_e = -(ll_p * torch.log(ll_p + 1e-10)).sum(dim=-1)
            int_scalars.append(ll_e.mean().item())

        # entropy drop / ratio
        first_e = int_scalars[2]  # logit_lens_entropy for first probe layer
        last_e = int_scalars[-1]  # logit_lens_entropy for last probe layer
        int_scalars.append(first_e - last_e)
        int_scalars.append(last_e / (first_e + 1e-10))

        internal_scalar_arr = np.array(int_scalars, dtype=np.float32)

        self._hidden.clear()
        return {
            "uncertainty": uncertainty_arr,
            "internal_scalars": internal_scalar_arr,
            "probe_vec": probe_vector,
        }

def preprocess(
    tokenizer: AutoTokenizer,
    prompt: str,
    answer: str,
) -> tuple[torch.Tensor, int]:
    messages_prompt = [{"role": "user", "content": prompt}]
    prompt_enc = tokenizer.apply_chat_template(
        messages_prompt,
        add_generation_prompt=True,
        tokenize=True,
    )
    prompt_token_ids = prompt_enc["input_ids"]
    answer_start_idx = len(prompt_token_ids)

    messages_full = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": answer},
    ]

    full_enc = tokenizer.apply_chat_template(
        messages_full,
        tokenize=True,
    )

    token_ids = full_enc["input_ids"]

    token_ids = torch.tensor([token_ids], dtype=torch.long)
    return token_ids, answer_start_idx

In [52]:
accumulator = FeatureAccumulator(model)
unc_list, int_list, probe_list, label_list = [], [], [], []

for _, row in tqdm(df_train.iterrows(), total=len(df_train)):
    token_ids, answer_start_idx = preprocess(tokenizer, row["prompt"], row["model_answer"])
    token_ids = token_ids.to(device)

    with accumulator:
        with torch.no_grad():
            outputs = model(token_ids)

    feats = accumulator.compute_features(
        outputs.logits,
        token_ids,
        answer_start_idx
    
    )
    
    del outputs
    torch.cuda.empty_cache()

    unc_list.append(feats["uncertainty"])
    int_list.append(feats["internal_scalars"])
    probe_list.append(feats["probe_vec"])
    label_list.append(row["is_hallucination"])

X_y_train = {
    "uncertainty_X":     np.stack(unc_list).astype(np.float32),
    "internal_scalar_X": np.stack(int_list).astype(np.float32),
    "probe_last_prompt": np.stack(probe_list).astype(np.float32),
    "labels":            np.array(label_list, dtype=np.int32),
}

100%|██████████| 731/731 [01:20<00:00,  9.10it/s]


## 3. Обучение классификатора

In [53]:
class HallucinationClassifier:
    def __init__(self):
        self.pca = PCA(n_components=64, random_state=42)
        self.scaler = StandardScaler()
        self.clf = LogisticRegression(
            C=1.0,
            max_iter=3000,
            class_weight="balanced",
            random_state=42,
        )
        self.threshold = 0.5

    def fit(self, X_y: dict):
        """Fit from pre-extracted features (npz)."""
        y = X_y["labels"]

        probe_last_prompt_pca = self.pca.fit_transform(
            X_y["probe_last_prompt"]
        )
        X = np.hstack([
            probe_last_prompt_pca,
            X_y["internal_scalar_X"],
            X_y["uncertainty_X"],
        ])

        X = self.scaler.fit_transform(X)
        self.clf.fit(X, y)

    def to_X(self, features: dict) -> np.ndarray:
        if features["probe_vec"].ndim == 1:
            return np.hstack([
                self.pca.transform(features["probe_vec"].reshape(1, -1)),
                features["internal_scalars"].reshape(1, -1),
                features["uncertainty"].reshape(1, -1),
            ])
        else:
            return np.hstack([
                self.pca.transform(features["probe_vec"]),
                features["internal_scalars"],
                features["uncertainty"],
            ])

    def predict_proba(self, features: dict) -> float:
        X_s = self.scaler.transform(self.to_X(features))
        return float(self.clf.predict_proba(X_s)[0, 1])

    def predict(self, features: dict) -> tuple[int, float]:
        prob = self.predict_proba(features)
        return int(prob >= self.threshold), prob

In [54]:
clf = HallucinationClassifier()
clf.fit(X_y_train)

## 4. Инференс

In [55]:
@dataclass
class ScoringResult:
    is_hallucination: bool
    is_hallucination_proba: float
    t_model_sec: float = 0.0
    t_overhead_sec: float = 0.0
    t_total_sec: float = 0.0


class GuardianOfTruth:
    def __init__(self, model, tokenizer, classifier: HallucinationClassifier):
        self.model = model
        self.tokenizer = tokenizer
        self.classifier = classifier
        self.accumulator = FeatureAccumulator(model)

    def score(self, prompt: str, answer: str) -> ScoringResult:
        token_ids, answer_start_idx = preprocess(self.tokenizer, prompt, answer)
        device = next(self.model.parameters()).device
        token_ids = token_ids.to(device)

        ############################
        #     Gigachat forward     #
        ############################
        t0 = time.perf_counter()
        with self.accumulator:
            with torch.no_grad():
                outputs = self.model(token_ids)
        t_model = time.perf_counter() - t0

        ########################
        #      Classifier      #
        ########################
        t1 = time.perf_counter()
        features = self.accumulator.compute_features(
            outputs.logits,
            token_ids,
            answer_start_idx,
        )
        del outputs
        torch.cuda.empty_cache()

        if features is None:
            return ScoringResult(is_hallucination=False, is_hallucination_proba=0.0)

        label, prob = self.classifier.predict(features)
        t_overhead = time.perf_counter() - t1

        return ScoringResult(
            is_hallucination=bool(label),
            is_hallucination_proba=prob,
            t_model_sec=t_model,
            t_overhead_sec=t_overhead,
            t_total_sec=t_model + t_overhead,
        )

## 5. Метрики

In [ ]:
guard = GuardianOfTruth(model, tokenizer, clf)

# ---------- train ----------
train_results = []
for _, row in tqdm(df_train.iterrows(), total=len(df_train), desc="train"):
    res = guard.score(row["prompt"], row["model_answer"])
    train_results.append(res)

df_train_scored = df_train.copy()
df_train_scored["pred_proba"] = [r.is_hallucination_proba for r in train_results]

# ---------- test ----------
test_results = []
for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="test"):
    res = guard.score(row["prompt"], row["model_answer"])
    test_results.append(res)

df_test_scored = df_test.copy()
df_test_scored["pred_proba"] = [r.is_hallucination_proba for r in test_results]
df_test_scored["t_model_ms"]    = [r.t_model_sec * 1000    for r in test_results]
df_test_scored["t_overhead_ms"] = [r.t_overhead_sec * 1000 for r in test_results]


metrics_df = pd.DataFrame([
    {'split': "Train", 'pr_auc': round(average_precision_score(
        df_train_scored["is_hallucination"],
        df_train_scored["pred_proba"].values),
        4)
    },
    {'split': "Test",  'pr_auc': round(average_precision_score(
        df_test_scored["is_hallucination"],
        df_test_scored["pred_proba"].values),
        4)
    },
])
metrics_df = metrics_df.set_index("split")
print(metrics_df.to_string())
print()
print("Timing (test, per sample):")
print(f"  model forward : {df_test_scored['t_model_ms'].mean():.1f} ms (mean)")
print(f"  overhead      : {df_test_scored['t_overhead_ms'].mean():.1f} ms (mean)")
print(f"  overhead p99  : {df_test_scored['t_overhead_ms'].quantile(0.99):.1f} ms")
overhead_ok = df_test_scored['t_overhead_ms'].quantile(0.99) < 1000
print(f"  {'PASS' if overhead_ok else 'FAIL'}: p99 overhead < 1000 ms")

       pr_auc
split        
Train  0.9171
Test   0.7638

Timing (test, per sample):
  model forward : 95.0 ms (mean)
  overhead      : 9.8 ms (mean)
  overhead p99  : 19.7 ms
  PASS: p99 overhead < 1000 ms
